# ML Testing & MLOps: Non-Regression Testing Before You Ship a Model

**Dataset:** [Palmer Archipelago (Antarctica) Penguins](https://allisonhorst.github.io/palmerpenguins/)
— 344 real morphological measurements (bill length/depth, flipper length, body mass) for three
penguin species (Adelie, Chinstrap, Gentoo), collected by Dr. Kristen Gorman at Palmer Station,
Antarctica LTER. Loaded at runtime from `mwaskom/seaborn-data`.

**Problem statement.** A wildlife-monitoring team has a `baseline` species-classifier in
production and wants to replace it with a `candidate` model that "looks better" on a held-out
test set. Your job is to be the QA gate for that swap: decide, with actual statistical rigor
instead of a single number, whether the candidate is *really* better, by how much, and whether
that improvement is large enough to justify the risk of swapping a production model — not just
whether it's nonzero. This is the same problem as deciding whether a flaky test's pass/fail is
signal or noise: one run is never evidence.

**What you should produce**, concretely:
1. A `paired_bootstrap_diff(y_true, pred_a, pred_b, metric_fn, n_boot, seed)` function that
   bootstraps the paired difference `metric_fn(y_true, pred_b) - metric_fn(y_true, pred_a)` by
   resampling the test set (with replacement), to get a distribution of the metric difference
   instead of one point estimate.
2. A `deployment_gate(diffs, min_improvement, alpha)` function returning a dict with: the
   `(1 - alpha)` two-sided percentile confidence interval, the mean diff, whether the result is
   **statistically significant** (CI excludes 0), whether it's **practically significant** (the
   CI's lower bound clears `min_improvement`), and a final `"deploy"` / `"hold"` decision —
   both conditions must hold to deploy.
3. Applying that gate to **accuracy** and **macro-F1** on one locked test set, comparing a
   regularized `LogisticRegression` baseline against a shallow `RandomForestClassifier`
   candidate.
4. A resplit experiment: re-run the train/test split 60 different ways (60 different
   `random_state` seeds), retrain both models fresh each time, and record how often the
   candidate's single-split point estimate "wins" on accuracy — to show how unstable a one-off
   comparison is.
5. A short written verdict on whether this candidate should ship, citing both the gate result
   and the resplit win-rate.


---

## Starter cells

Fill in every `# TODO`. Don't look at the solution until you've tried each step.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

# This is the split used ONLY for the final deploy/hold decision below -- never for
# tuning -- so the gate's answer isn't contaminated by peeking at the test set.
DECISION_SPLIT_SEED = 73
MIN_IMPROVEMENT = 0.02  # candidate must clear baseline by at least 2pp to be worth the swap risk

In [ ]:
PENGUINS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins = pd.read_csv(PENGUINS_URL)

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
TARGET = "species"

# TODO: this is real field data collected by hand -- it has missing values (e.g. in `sex`,
# which you aren't using, and a handful of rows missing the numeric features you ARE using).
# Drop rows with any missing value in FEATURES + TARGET and reset the index.
penguins_clean = ...

print(f"rows before: {len(penguins)}, after dropping incomplete rows: {len(penguins_clean)}")
print(penguins_clean[TARGET].value_counts())

X = ...  # TODO: penguins_clean[FEATURES]
y = ...  # TODO: penguins_clean[TARGET]

In [ ]:
# TODO: stratified train/test split using DECISION_SPLIT_SEED, test_size=0.3
X_train, X_test, y_train, y_test = ...


def make_baseline():
    # TODO: a Pipeline of StandardScaler + LogisticRegression(max_iter=1000, C=0.08, random_state=42)
    ...


def make_candidate():
    # TODO: a RandomForestClassifier(n_estimators=30, max_depth=3, random_state=7)
    ...


# TODO: fit both on (X_train, y_train); predict on X_test
baseline = ...
candidate = ...
pred_baseline = ...
pred_candidate = ...

In [ ]:
# TODO: compute accuracy_score and f1_score(average="macro") for both models on y_test
acc_baseline, acc_candidate = ..., ...
f1_baseline, f1_candidate = ..., ...

print(f"accuracy  baseline={acc_baseline:.3f}  candidate={acc_candidate:.3f}  diff={acc_candidate - acc_baseline:+.3f}")
print(f"macro-F1  baseline={f1_baseline:.3f}  candidate={f1_candidate:.3f}  diff={f1_candidate - f1_baseline:+.3f}")
# Before running the gate below: would YOU ship based on this alone? Keep your answer in mind.

In [ ]:
def paired_bootstrap_diff(y_true, pred_a, pred_b, metric_fn, n_boot=2000, seed=0):
    """
    Resample (y_true, pred_a, pred_b) together, with replacement, `n_boot` times
    (same size as y_true each time), and return an array of
    metric_fn(resampled_y_true, resampled_pred_b) - metric_fn(resampled_y_true, resampled_pred_a).
    """
    # TODO: implement using a seeded np.random.default_rng
    ...

In [ ]:
def deployment_gate(diffs, min_improvement=0.0, alpha=0.05):
    """
    Given an array of bootstrap diffs (candidate - baseline), return a dict with:
      ci_low, ci_high            -- the (1 - alpha) two-sided percentile CI of `diffs`
      mean_diff                  -- diffs.mean()
      statistically_significant  -- True if the CI excludes 0
      practically_significant    -- True if ci_low >= min_improvement
      decision                   -- "deploy" if both flags above are True, else "hold"
    """
    # TODO: implement
    ...

In [ ]:
# TODO: run paired_bootstrap_diff + deployment_gate for accuracy, then for macro-F1,
# using MIN_IMPROVEMENT, and print each gate's dict.
acc_diffs = ...
acc_gate = ...

f1_diffs = ...
f1_gate = ...

print("accuracy gate:", acc_gate)
print("macro-F1 gate:", f1_gate)

In [ ]:
# TODO: resplit experiment. For each seed in range(60):
#   - fresh stratified train/test split of the FULL (X, y) with that seed, test_size=0.3
#   - fresh baseline and candidate (fit on that split's train set)
#   - record whether candidate's point-estimate accuracy beat baseline's on that split's test set
# Report the fraction of the 60 splits where candidate "won".
candidate_win_rate = ...
print(f"candidate win rate across 60 independent resplits: {candidate_win_rate:.2%}")

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

DECISION_SPLIT_SEED = 73
MIN_IMPROVEMENT = 0.02

PENGUINS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins = pd.read_csv(PENGUINS_URL)

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
TARGET = "species"

penguins_clean = penguins.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
print(f"rows before: {len(penguins)}, after dropping incomplete rows: {len(penguins_clean)}")
print(penguins_clean[TARGET].value_counts())

X = penguins_clean[FEATURES]
y = penguins_clean[TARGET]

# --- 1. the locked decision split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=DECISION_SPLIT_SEED
)

def make_baseline():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=0.08, random_state=42))

def make_candidate():
    return RandomForestClassifier(n_estimators=30, max_depth=3, random_state=7)

baseline = make_baseline().fit(X_train, y_train)
candidate = make_candidate().fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)
pred_candidate = candidate.predict(X_test)

# --- 2. point estimate -- what a naive comparison would show ---
acc_baseline = accuracy_score(y_test, pred_baseline)
acc_candidate = accuracy_score(y_test, pred_candidate)
f1_baseline = f1_score(y_test, pred_baseline, average="macro")
f1_candidate = f1_score(y_test, pred_candidate, average="macro")

print(f"accuracy  baseline={acc_baseline:.3f}  candidate={acc_candidate:.3f}  diff={acc_candidate - acc_baseline:+.3f}")
print(f"macro-F1  baseline={f1_baseline:.3f}  candidate={f1_candidate:.3f}  diff={f1_candidate - f1_baseline:+.3f}")
# On this split the candidate looks clearly better: +0.019 accuracy, +0.027 macro-F1.
# A naive engineer stops here and ships. The rest of this notebook shows why that's premature.

# --- 3. bootstrap the paired difference instead of trusting one number ---
def paired_bootstrap_diff(y_true, pred_a, pred_b, metric_fn, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    n = len(y_true)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = metric_fn(y_true[idx], pred_b[idx]) - metric_fn(y_true[idx], pred_a[idx])
    return diffs

def deployment_gate(diffs, min_improvement=0.0, alpha=0.05):
    ci_low, ci_high = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    mean_diff = diffs.mean()
    statistically_significant = ci_low > 0 or ci_high < 0
    practically_significant = ci_low >= min_improvement
    decision = "deploy" if (statistically_significant and practically_significant) else "hold"
    return {
        "ci_low": ci_low,
        "ci_high": ci_high,
        "mean_diff": mean_diff,
        "statistically_significant": statistically_significant,
        "practically_significant": practically_significant,
        "decision": decision,
    }

acc_diffs = paired_bootstrap_diff(y_test, pred_baseline, pred_candidate, accuracy_score, seed=1)
acc_gate = deployment_gate(acc_diffs, min_improvement=MIN_IMPROVEMENT)

f1_diffs = paired_bootstrap_diff(
    y_test, pred_baseline, pred_candidate, lambda a, b: f1_score(a, b, average="macro"), seed=2
)
f1_gate = deployment_gate(f1_diffs, min_improvement=MIN_IMPROVEMENT)

print("accuracy gate:", acc_gate)
print("macro-F1 gate:", f1_gate)
# The 95% CI for the accuracy diff is roughly [-0.02, 0.06]: it comfortably includes zero,
# so the apparent +0.019 improvement is not statistically significant, and its lower bound
# is nowhere near MIN_IMPROVEMENT=0.02 either. The gate correctly returns "hold" even though
# the point estimate alone looked like a clean win.

# --- 4. resplit experiment: is this split's "win" reproducible? ---
wins = []
for seed in range(60):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    b = make_baseline().fit(Xtr, ytr)
    c = make_candidate().fit(Xtr, ytr)
    wins.append(accuracy_score(yte, c.predict(Xte)) > accuracy_score(yte, b.predict(Xte)))

candidate_win_rate = float(np.mean(wins))
print(f"candidate win rate across 60 independent resplits: {candidate_win_rate:.2%}")
# The candidate wins on only ~8% of resplits -- the "win" on the decision split was a fluke
# of which rows landed in that particular test set, not a real, repeatable improvement.
```

**Verdict:** hold. The single-split point estimate (+1.9pp accuracy) is exactly the kind of
number that tempts a team to ship immediately, but the bootstrap CI shows that difference is
barely distinguishable from zero, and the 60-way resplit experiment shows the candidate only
"wins" about 1 time in 12 — most splits actually favor the baseline. Shipping on the point
estimate alone would have been a regression in disguise roughly 90% of the time.

**Why this matters for ML quality engineering:** a model comparison is a statistical claim, not
a single measurement — exactly like a flaky test is a signal you have to characterize across
runs, not trust on one pass/fail. `paired_bootstrap_diff` turns "the candidate scored higher"
into a distribution you can reason about. `deployment_gate` encodes the same two-part logic a
good SQA release gate uses: is the effect *real* (statistically significant) **and** is it *big
enough to matter* (practically significant, i.e. worth the deploy risk)? Collapsing those into
one point estimate — the way most ad-hoc model comparisons are done — is how silent regressions
and overstated "wins" both slip into production.

</details>
